# Praktikum: Unsupervised Learning

## 🎯 Maqsad
Ushbu praktikumda siz K-means, Hierarchical Clustering va PCA'ni real datasetlarda qo'llaysiz.

## 📋 Vazifalar
1. Mall Customers Segmentation (K-means)
2. Country Data Clustering (Hierarchical)
3. Digits Dataset (PCA + Clustering)

---

In [ ]:
# Kerakli kutubxonalar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.datasets import load_digits
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Setup complete!")

---

# Mashq 1: Mall Customers Segmentation

## 📊 Dataset: Mall Customers

Savdo markazining mijozlari haqida ma'lumot:
- **CustomerID**: Mijoz ID
- **Gender**: Jins (Male/Female)
- **Age**: Yosh
- **Annual Income**: Yillik daromad ($1000)
- **Spending Score**: Xarajat ko'rsatkichi (1-100)

### ✏️ Vazifa:
1. Ma'lumotlarni yuklash va EDA
2. Optimal K ni topish (Elbow Method)
3. K-means clustering
4. Natijalarni vizualizatsiya qilish
5. Har bir klasterning xususiyatlarini tahlil qilish

In [ ]:
# Synthetic Mall Customers data
np.random.seed(42)
n = 200

# 5 ta segment yaratish
# Segment 1: Young, low income, low spending
seg1_age = np.random.normal(25, 5, 40)
seg1_income = np.random.normal(30, 5, 40)
seg1_spending = np.random.normal(30, 5, 40)

# Segment 2: Young, low income, high spending
seg2_age = np.random.normal(25, 5, 40)
seg2_income = np.random.normal(30, 5, 40)
seg2_spending = np.random.normal(75, 5, 40)

# Segment 3: Middle age, high income, high spending
seg3_age = np.random.normal(45, 5, 40)
seg3_income = np.random.normal(80, 5, 40)
seg3_spending = np.random.normal(75, 5, 40)

# Segment 4: Middle age, high income, low spending
seg4_age = np.random.normal(45, 5, 40)
seg4_income = np.random.normal(80, 5, 40)
seg4_spending = np.random.normal(30, 5, 40)

# Segment 5: Old, medium income, medium spending
seg5_age = np.random.normal(60, 5, 40)
seg5_income = np.random.normal(55, 5, 40)
seg5_spending = np.random.normal(50, 5, 40)

# Combine
age = np.concatenate([seg1_age, seg2_age, seg3_age, seg4_age, seg5_age])
income = np.concatenate([seg1_income, seg2_income, seg3_income, seg4_income, seg5_income])
spending = np.concatenate([seg1_spending, seg2_spending, seg3_spending, seg4_spending, seg5_spending])

# Clip values
age = np.clip(age, 18, 80)
income = np.clip(income, 15, 150)
spending = np.clip(spending, 1, 99)

# Gender (random)
gender = np.random.choice(['Male', 'Female'], n)

# DataFrame
df_mall = pd.DataFrame({
    'CustomerID': range(1, n+1),
    'Gender': gender,
    'Age': age.astype(int),
    'Annual Income (k$)': income.round(1),
    'Spending Score (1-100)': spending.astype(int)
})

print("Mall Customers Dataset:")
print(df_mall.head(10))
print(f"\nShape: {df_mall.shape}")
print(f"\nInfo:")
print(df_mall.describe())

### 📊 Step 1: Exploratory Data Analysis (EDA)

In [ ]:
# EDA: Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Age distribution
axes[0, 0].hist(df_mall['Age'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Age', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Age Distribution', fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Income distribution
axes[0, 1].hist(df_mall['Annual Income (k$)'], bins=20, color='green', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Annual Income (k$)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Income Distribution', fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Spending distribution
axes[1, 0].hist(df_mall['Spending Score (1-100)'], bins=20, color='orange', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Spending Score', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Spending Score Distribution', fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Income vs Spending scatter
axes[1, 1].scatter(df_mall['Annual Income (k$)'], df_mall['Spending Score (1-100)'],
                   s=50, alpha=0.6, c='purple', edgecolors='k')
axes[1, 1].set_xlabel('Annual Income (k$)', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Spending Score (1-100)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Income vs Spending', fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 EDA Insights:")
print(f"  - Age range: {df_mall['Age'].min()} - {df_mall['Age'].max()}")
print(f"  - Income range: ${df_mall['Annual Income (k$)'].min():.1f}k - ${df_mall['Annual Income (k$)'].max():.1f}k")
print(f"  - Spending range: {df_mall['Spending Score (1-100)'].min()} - {df_mall['Spending Score (1-100)'].max()}")
print(f"  - Gender: {df_mall['Gender'].value_counts().to_dict()}")

### 🔍 Step 2: Optimal K ni topish (Elbow Method)

**Vazifa**: Income va Spending Score'ni ishlatib K-means clustering qiling.

In [ ]:
# TODO: Income va Spending Score'ni tanlab X yasang
X_mall = df_mall[['Annual Income (k$)', 'Spending Score (1-100)']].values

# TODO: Standardization
scaler_mall = StandardScaler()
X_mall_scaled = scaler_mall.fit_transform(X_mall)

# TODO: Elbow Method - K=2 dan 10 gacha
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_mall_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_mall_scaled, kmeans.labels_))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Inertia
axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=10)
axes[0].set_xlabel('K (Number of Clusters)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Inertia', fontsize=12, fontweight='bold')
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Silhouette
axes[1].plot(K_range, silhouettes, 'ro-', linewidth=2, markersize=10)
axes[1].set_xlabel('K (Number of Clusters)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_title('Silhouette Score', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# TODO: Optimal K ni tanlang (elbow va silhouette'ga qarab)
optimal_k = K_range[np.argmax(silhouettes)]
print(f"\n🎯 Optimal K: {optimal_k}")
print(f"   Best Silhouette Score: {max(silhouettes):.4f}")

### 🎨 Step 3: K-means Clustering

In [ ]:
# TODO: Optimal K bilan K-means clustering
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_mall['Cluster'] = kmeans_final.fit_predict(X_mall_scaled)

# Visualization
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df_mall['Annual Income (k$)'], df_mall['Spending Score (1-100)'],
                      c=df_mall['Cluster'], s=100, alpha=0.7, cmap='viridis',
                      edgecolors='k', linewidth=1)

# Centroids
centroids = scaler_mall.inverse_transform(kmeans_final.cluster_centers_)
plt.scatter(centroids[:, 0], centroids[:, 1], s=400, marker='X',
            c='red', edgecolors='black', linewidths=3, label='Centroids')

plt.xlabel('Annual Income (k$)', fontsize=13, fontweight='bold')
plt.ylabel('Spending Score (1-100)', fontsize=13, fontweight='bold')
plt.title(f'Customer Segmentation (K={optimal_k})', fontsize=15, fontweight='bold')
plt.colorbar(scatter, label='Cluster')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Clustering Complete!")

### 📈 Step 4: Cluster Analysis

In [ ]:
# TODO: Har bir klasterning statistikasini hisoblang
cluster_stats = df_mall.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean()
cluster_counts = df_mall['Cluster'].value_counts().sort_index()

print("\n" + "="*70)
print("CUSTOMER SEGMENTATION ANALYSIS")
print("="*70)

for cluster in range(optimal_k):
    print(f"\n📍 Cluster {cluster}: {cluster_counts[cluster]} customers")
    print(f"   Avg Age: {cluster_stats.loc[cluster, 'Age']:.1f} years")
    print(f"   Avg Income: ${cluster_stats.loc[cluster, 'Annual Income (k$)']:.1f}k")
    print(f"   Avg Spending: {cluster_stats.loc[cluster, 'Spending Score (1-100)']:.1f}")
    
    # Interpretation
    income = cluster_stats.loc[cluster, 'Annual Income (k$)']
    spending = cluster_stats.loc[cluster, 'Spending Score (1-100)']
    
    if income > 60 and spending > 60:
        segment = "💰 High Income, High Spending (Target Group)"
    elif income > 60 and spending < 40:
        segment = "💼 High Income, Low Spending (Careful Spenders)"
    elif income < 40 and spending > 60:
        segment = "🛍️ Low Income, High Spending (Impulsive Buyers)"
    elif income < 40 and spending < 40:
        segment = "💸 Low Income, Low Spending (Budget Conscious)"
    else:
        segment = "⚖️ Medium Income, Medium Spending (Average)"
    
    print(f"   Segment: {segment}")

print("\n" + "="*70)

# TODO: Gender distribution per cluster
gender_dist = df_mall.groupby(['Cluster', 'Gender']).size().unstack(fill_value=0)
print("\n📊 Gender Distribution per Cluster:")
print(gender_dist)

### ✅ Mashq 1 Complete!

**Nima qildik:**
- ✅ EDA - ma'lumotlarni tahlil qildik
- ✅ Elbow Method - optimal K ni topdik
- ✅ K-means - clustering qildik
- ✅ Cluster Analysis - har bir segmentni tushuntirdik

---

# Mashq 2: Country Data - Hierarchical Clustering

## 📊 Dataset: Country Socio-Economic Data

Mamlakatlar haqida ma'lumot:
- **GDP per capita**: Aholi boshiga YaIM
- **Life expectancy**: O'rtacha umr davomiyligi
- **Child mortality**: Bola o'limi (1000 tug'ilishda)
- **Literacy rate**: Savodxonlik darajasi

### ✏️ Vazifa:
1. Synthetic data yaratish
2. Dendrogram chizish
3. Hierarchical clustering (turli linkage metodlar)
4. Mamlakatlarni guruhlash

In [ ]:
# TODO: Synthetic country data
np.random.seed(42)
n_countries = 50

# 3 ta guruh: Developed, Developing, Underdeveloped
# Developed (15 countries)
dev_gdp = np.random.normal(50000, 10000, 15)
dev_life = np.random.normal(80, 3, 15)
dev_mortality = np.random.normal(5, 1, 15)
dev_literacy = np.random.normal(98, 1, 15)

# Developing (20 countries)
ing_gdp = np.random.normal(15000, 5000, 20)
ing_life = np.random.normal(70, 4, 20)
ing_mortality = np.random.normal(30, 5, 20)
ing_literacy = np.random.normal(80, 5, 20)

# Underdeveloped (15 countries)
under_gdp = np.random.normal(3000, 1000, 15)
under_life = np.random.normal(55, 5, 15)
under_mortality = np.random.normal(70, 10, 15)
under_literacy = np.random.normal(50, 10, 15)

# Combine
gdp = np.concatenate([dev_gdp, ing_gdp, under_gdp])
life = np.concatenate([dev_life, ing_life, under_life])
mortality = np.concatenate([dev_mortality, ing_mortality, under_mortality])
literacy = np.concatenate([dev_literacy, ing_literacy, under_literacy])

# Clip
gdp = np.clip(gdp, 500, 100000)
life = np.clip(life, 40, 90)
mortality = np.clip(mortality, 1, 150)
literacy = np.clip(literacy, 20, 100)

# DataFrame
df_country = pd.DataFrame({
    'Country': [f'Country_{i+1}' for i in range(n_countries)],
    'GDP_per_capita': gdp.round(0),
    'Life_expectancy': life.round(1),
    'Child_mortality': mortality.round(1),
    'Literacy_rate': literacy.round(1)
})

print("Country Socio-Economic Dataset:")
print(df_country.head(10))
print(f"\nShape: {df_country.shape}")
print(f"\nStatistics:")
print(df_country.describe())

### 🌳 Step 1: Dendrogram

In [ ]:
# TODO: Feature selection va standardization
X_country = df_country[['GDP_per_capita', 'Life_expectancy', 'Child_mortality', 'Literacy_rate']].values

scaler_country = StandardScaler()
X_country_scaled = scaler_country.fit_transform(X_country)

# TODO: Linkage matrix (Ward method)
linkage_mat = linkage(X_country_scaled, method='ward')

# Dendrogram
plt.figure(figsize=(16, 8))
dendrogram(linkage_mat, labels=df_country['Country'].values, leaf_font_size=8)
plt.xlabel('Country', fontsize=12, fontweight='bold')
plt.ylabel('Distance', fontsize=12, fontweight='bold')
plt.title('Hierarchical Clustering - Dendrogram (Ward Linkage)', fontsize=14, fontweight='bold')
plt.axhline(y=10, color='red', linestyle='--', linewidth=2, label='Cut threshold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n🌳 Dendrogram chizildi!")
print("💡 Qizil chiziq: Clustering uchun 'cut' qilingan joy")

### 🔗 Step 2: Turli Linkage Metodlarni Taqqoslash

In [ ]:
# TODO: Single, Complete, Average, Ward linkage'larni test qiling
linkage_methods = ['single', 'complete', 'average', 'ward']
n_clusters = 3

results = {}
for method in linkage_methods:
    hier = AgglomerativeClustering(n_clusters=n_clusters, linkage=method)
    labels = hier.fit_predict(X_country_scaled)
    sil_score = silhouette_score(X_country_scaled, labels)
    results[method] = {'labels': labels, 'silhouette': sil_score}

# Natijalarni chiqarish
print("\n" + "="*50)
print("LINKAGE METHODS COMPARISON")
print("="*50)
for method, res in results.items():
    print(f"{method.capitalize():15} Silhouette: {res['silhouette']:.4f}")
print("="*50)

# Eng yaxshi method
best_method = max(results, key=lambda x: results[x]['silhouette'])
print(f"\n🏆 Best Linkage: {best_method.capitalize()} (Silhouette: {results[best_method]['silhouette']:.4f})")

### 📊 Step 3: Clustering Result (PCA 2D visualization)

In [ ]:
# TODO: PCA: 4D → 2D
pca_country = PCA(n_components=2)
X_country_pca = pca_country.fit_transform(X_country_scaled)

# Best linkage method bilan clustering
df_country['Cluster'] = results[best_method]['labels']

# Visualization
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_country_pca[:, 0], X_country_pca[:, 1], 
                      c=df_country['Cluster'], s=120, alpha=0.7, 
                      cmap='viridis', edgecolors='k', linewidth=1)

# Country labels
for i, country in enumerate(df_country['Country']):
    plt.annotate(country, (X_country_pca[i, 0], X_country_pca[i, 1]),
                 fontsize=7, alpha=0.7)

plt.xlabel(f'PC1 ({pca_country.explained_variance_ratio_[0]:.1%})', fontsize=12, fontweight='bold')
plt.ylabel(f'PC2 ({pca_country.explained_variance_ratio_[1]:.1%})', fontsize=12, fontweight='bold')
plt.title(f'Country Clustering ({best_method.capitalize()} Linkage, {n_clusters} clusters)', 
          fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✅ Hierarchical Clustering complete ({best_method} linkage)!")

### 📈 Step 4: Cluster Interpretation

In [ ]:
# TODO: Har bir klasterning xususiyatlarini tahlil qiling
cluster_country_stats = df_country.groupby('Cluster')[['GDP_per_capita', 'Life_expectancy', 
                                                         'Child_mortality', 'Literacy_rate']].mean()
cluster_country_counts = df_country['Cluster'].value_counts().sort_index()

print("\n" + "="*70)
print("COUNTRY CLUSTERING ANALYSIS")
print("="*70)

for cluster in range(n_clusters):
    print(f"\n📍 Cluster {cluster}: {cluster_country_counts[cluster]} countries")
    print(f"   Avg GDP per capita: ${cluster_country_stats.loc[cluster, 'GDP_per_capita']:.0f}")
    print(f"   Avg Life expectancy: {cluster_country_stats.loc[cluster, 'Life_expectancy']:.1f} years")
    print(f"   Avg Child mortality: {cluster_country_stats.loc[cluster, 'Child_mortality']:.1f} per 1000")
    print(f"   Avg Literacy rate: {cluster_country_stats.loc[cluster, 'Literacy_rate']:.1f}%")
    
    # Countries in this cluster
    countries_in_cluster = df_country[df_country['Cluster'] == cluster]['Country'].tolist()
    print(f"   Countries: {', '.join(countries_in_cluster[:5])}...")

print("\n" + "="*70)

### ✅ Mashq 2 Complete!

**Nima qildik:**
- ✅ Dendrogram chizish
- ✅ Turli linkage metodlarni taqqoslash
- ✅ Hierarchical clustering
- ✅ PCA bilan visualization
- ✅ Mamlakatlarni guruhlash

---

# Mashq 3: Digits Dataset - PCA + Clustering

## 📊 Dataset: Handwritten Digits (8x8)

Sklearn'ning digits dataseti:
- **Images**: 8x8 pixel raqamlar (0-9)
- **Features**: 64 (8x8 pixels)
- **Target**: Raqam (0-9)

### ✏️ Vazifa:
1. Datasetni yuklash
2. PCA: 64D → 2D
3. Visualization
4. Clustering (K-means)
5. Natijalarni taqqoslash

In [ ]:
# TODO: Digits datasetini yuklash
digits = load_digits()
X_digits = digits.data  # 64 features (8x8 pixels)
y_digits = digits.target  # True labels (0-9)

print(f"Digits Dataset:")
print(f"  Shape: {X_digits.shape}")
print(f"  Classes: {np.unique(y_digits)}")
print(f"  Features: {X_digits.shape[1]} (8x8 pixels)")

# Bir nechta raqamlarni ko'rsatish
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Label: {y_digits[i]}', fontsize=11, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample Digit Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 🔍 Step 1: PCA - Dimensionality Reduction (64D → 2D)

In [ ]:
# TODO: Standardization
scaler_digits = StandardScaler()
X_digits_scaled = scaler_digits.fit_transform(X_digits)

# TODO: PCA: 64D → 2D
pca_digits = PCA(n_components=2)
X_digits_pca = pca_digits.fit_transform(X_digits_scaled)

print(f"\n📊 PCA: {X_digits.shape[1]}D → {pca_digits.n_components_}D")
print(f"   PC1 variance: {pca_digits.explained_variance_ratio_[0]:.2%}")
print(f"   PC2 variance: {pca_digits.explained_variance_ratio_[1]:.2%}")
print(f"   Total variance: {pca_digits.explained_variance_ratio_.sum():.2%}")

# Visualization: True labels
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_digits_pca[:, 0], X_digits_pca[:, 1], 
                      c=y_digits, s=50, alpha=0.6, cmap='tab10', edgecolors='k', linewidth=0.5)
plt.xlabel(f'PC1 ({pca_digits.explained_variance_ratio_[0]:.1%})', fontsize=12, fontweight='bold')
plt.ylabel(f'PC2 ({pca_digits.explained_variance_ratio_[1]:.1%})', fontsize=12, fontweight='bold')
plt.title('Digits Dataset - PCA 2D (True Labels)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Digit', ticks=range(10))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🎨 Step 2: Scree Plot - Nechta PC kerak?

In [ ]:
# TODO: Barcha PC'larni hisoblash
pca_all_digits = PCA()
pca_all_digits.fit(X_digits_scaled)

exp_var = pca_all_digits.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

# Scree Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Individual variance
axes[0].bar(range(1, 21), exp_var[:20], alpha=0.7, color='steelblue', edgecolor='black')
axes[0].plot(range(1, 21), exp_var[:20], 'ro-', linewidth=2, markersize=8)
axes[0].set_xlabel('Principal Component', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Explained Variance Ratio', fontsize=12, fontweight='bold')
axes[0].set_title('Scree Plot (Top 20 PCs)', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Cumulative variance
axes[1].plot(range(1, 21), cum_var[:20], 'bo-', linewidth=2, markersize=8)
axes[1].axhline(y=0.95, color='red', linestyle='--', linewidth=2, label='95% threshold')
axes[1].set_xlabel('Number of Components', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Cumulative Variance', fontsize=12, fontweight='bold')
axes[1].set_title('Cumulative Explained Variance', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# TODO: 95% variance uchun nechta PC kerak?
n_components_95 = np.argmax(cum_var >= 0.95) + 1
print(f"\n🎯 95% variance uchun {n_components_95} ta PC kerak")
print(f"   2 PC bilan faqat {cum_var[1]:.2%} variance saqlanadi")

### 🔄 Step 3: K-means Clustering (PCA space'da)

In [ ]:
# TODO: K-means clustering (K=10, chunki 10 ta digit)
kmeans_digits = KMeans(n_clusters=10, random_state=42, n_init=10)
y_pred_digits = kmeans_digits.fit_predict(X_digits_pca)

# Visualization: Predicted clusters
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_digits_pca[:, 0], X_digits_pca[:, 1],
                      c=y_pred_digits, s=50, alpha=0.6, cmap='tab10',
                      edgecolors='k', linewidth=0.5)
plt.scatter(kmeans_digits.cluster_centers_[:, 0], kmeans_digits.cluster_centers_[:, 1],
            s=300, marker='X', c='red', edgecolors='black', linewidths=2, label='Centroids')
plt.xlabel(f'PC1 ({pca_digits.explained_variance_ratio_[0]:.1%})', fontsize=12, fontweight='bold')
plt.ylabel(f'PC2 ({pca_digits.explained_variance_ratio_[1]:.1%})', fontsize=12, fontweight='bold')
plt.title('Digits Dataset - K-means Clustering (K=10)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster', ticks=range(10))
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Silhouette Score
sil_digits = silhouette_score(X_digits_pca, y_pred_digits)
print(f"\n📊 Silhouette Score: {sil_digits:.4f}")

### 📊 Step 4: Comparison - True Labels vs Predicted Clusters

In [ ]:
# TODO: Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# True labels
scatter1 = axes[0].scatter(X_digits_pca[:, 0], X_digits_pca[:, 1],
                           c=y_digits, s=50, alpha=0.6, cmap='tab10',
                           edgecolors='k', linewidth=0.5)
axes[0].set_xlabel('PC1', fontsize=12, fontweight='bold')
axes[0].set_ylabel('PC2', fontsize=12, fontweight='bold')
axes[0].set_title('True Labels', fontsize=14, fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='Digit', ticks=range(10))
axes[0].grid(True, alpha=0.3)

# Predicted clusters
scatter2 = axes[1].scatter(X_digits_pca[:, 0], X_digits_pca[:, 1],
                           c=y_pred_digits, s=50, alpha=0.6, cmap='tab10',
                           edgecolors='k', linewidth=0.5)
axes[1].scatter(kmeans_digits.cluster_centers_[:, 0], kmeans_digits.cluster_centers_[:, 1],
                s=300, marker='X', c='red', edgecolors='black', linewidths=2)
axes[1].set_xlabel('PC1', fontsize=12, fontweight='bold')
axes[1].set_ylabel('PC2', fontsize=12, fontweight='bold')
axes[1].set_title('K-means Clusters', fontsize=14, fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Cluster', ticks=range(10))
axes[1].grid(True, alpha=0.3)

plt.suptitle('Digits: True Labels vs Predicted Clusters', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Observation:")
print("  - K-means unsupervised (label yo'q) lekin yaxshi ajratadi")
print("  - Ba'zi raqamlar overlap qiladi (masalan, 3 va 8)")
print(f"  - PCA: {cum_var[1]:.2%} variance bilan vizualizatsiya")

### ✅ Mashq 3 Complete!

**Nima qildik:**
- ✅ PCA: 64D → 2D dimensionality reduction
- ✅ Scree Plot: Optimal PC tanlash
- ✅ K-means clustering (unsupervised)
- ✅ True labels vs Predicted clusters comparison

---

# 🎉 Praktikum Tugadi!

## ✅ Nima o'rgandik:

### Mashq 1: Mall Customers
- K-means clustering
- Elbow Method (optimal K)
- Customer segmentation
- Cluster interpretation

### Mashq 2: Country Data
- Hierarchical clustering
- Dendrogram
- Linkage methods comparison
- PCA visualization

### Mashq 3: Digits Dataset
- PCA (dimensionality reduction)
- Scree Plot
- High-dimensional clustering
- Unsupervised learning evaluation

---

## 🚀 Keyingi Qadamlar:
1. **homework.md** - Uyga vazifani ishlang
2. **unsupervised_guide.md** - Qo'llanmani o'qing
3. Real datasetlar bilan mashq qiling!

**Good luck! 🎯**